In [35]:
import pandas as pd
import random
from datetime import datetime
from pyspark.sql import SparkSession                    
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType
import pandas as pd
import random
from datetime import datetime
import pytz

StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 36, Finished, Available, Finished)

In [23]:


# Define device to store/location mapping
store_mapping = {
    (1, 10): ("Miami", "S001"),
    (11, 20): ("San Diego", "S002"),
    (21, 30): ("New York", "S003"),
    (31, 40): ("San Francisco", "S004"),
    (41, 50): ("Las Vegas", "S005"),
    (51, 60): ("London", "S006"),
    (61, 70): ("Dubai", "S007"),
    (71, 80): ("Singapore", "S008"),
    (81, 90): ("Tokyo", "S009"),
    (91, 100): ("Chicago", "S010"),
}

# Get current UTC time
now_utc = datetime.utcnow()

# Generate 100 records
records = []
for i in range(1, 101):
    device_id = f"TH{i:03}"
    # Determine store and city
    for (start, end), (city, store) in store_mapping.items():
        if start <= i <= end:
            city_name = city
            store_id = store
            break
            
    # Set values

    battery = random.randint(0, 100) 
    temp = round(random.uniform(60.0, 74.0), 1)
    temp_uom = "F"
    enqueued_time = now_utc
    event_processed_time = now_utc
    event_enqueued_time = now_utc
    partition_id = random.randint(1, 4)

    records.append({
        "DeviceId": device_id,
        "City": city_name,
        "StoreID": store_id,
        "EnqueuedTimeUTC": enqueued_time,
        "BatteryLevel": battery,
        "Temp": temp,
        "Temp_UoM": temp_uom
    })



# Create DataFrame
df = pd.DataFrame(records)

# Show first few rows
print(df.head())

# Optionally save to CSV
# df.to_csv("thermostat_realtime_data.csv", index=False)


StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 24, Finished, Available, Finished)

  DeviceId   City StoreID            EnqueuedTimeUTC  BatteryLevel  Temp  \
0    TH001  Miami    S001 2025-08-22 11:03:31.052917            78  68.8   
1    TH002  Miami    S001 2025-08-22 11:03:31.052917            55  65.3   
2    TH003  Miami    S001 2025-08-22 11:03:31.052917            90  63.6   
3    TH004  Miami    S001 2025-08-22 11:03:31.052917            75  63.7   
4    TH005  Miami    S001 2025-08-22 11:03:31.052917            70  66.0   

  Temp_UoM  
0        F  
1        F  
2        F  
3        F  
4        F  


In [36]:
import random
import pandas as pd
from datetime import datetime

# Define device to store/location mapping
store_mapping = {
    (1, 10): ("Miami", "S001"),
    (11, 20): ("San Diego", "S002"),
    (21, 30): ("New York", "S003"),
    (31, 40): ("San Francisco", "S004"),
    (41, 50): ("Las Vegas", "S005"),
    (51, 60): ("London", "S006"),
    (61, 70): ("Dubai", "S007"),
    (71, 80): ("Singapore", "S008"),
    (81, 90): ("Tokyo", "S009"),
    (91, 100): ("Chicago", "S010"),
}

# Shift mapping (hour → shift no)
def get_shift(hour):
    if 6 <= hour < 14:
        return 1
    elif 14 <= hour < 22:
        return 2
    else:  # 22:00–06:00
        return 3

# Get current UTC time
now_utc = datetime.utcnow()

# Generate 100 records
records = []
for i in range(1, 101):
    device_id = f"TH{i:03}"
    # Determine store and city
    for (start, end), (city, store) in store_mapping.items():
        if start <= i <= end:
            city_name = city
            store_id = store
            break
            
    # Set values
    battery = random.randint(0, 100) 
    temp = round(random.uniform(60.0, 74.0), 1)
    temp_uom = "F"
    enqueued_time = now_utc
    partition_id = random.randint(1, 4)

    # Find shift number based on hour
    shift_no = get_shift(enqueued_time.hour)
    combined_id = f"{store_id}-{shift_no}"

    records.append({
        "DeviceId": device_id,
        "City": city_name,
        "StoreID": store_id,
        "EnqueuedTimeUTC": enqueued_time,
        "BatteryLevel": battery,
        "Temp": temp,
        "Temp_UoM": temp_uom,
        "CombinedID": combined_id
    })

# Create DataFrame
df = pd.DataFrame(records)

# Show first few rows
print(df.head())


StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 37, Finished, Available, Finished)

  DeviceId   City StoreID            EnqueuedTimeUTC  BatteryLevel  Temp  \
0    TH001  Miami    S001 2025-08-22 11:28:39.390101            59  71.2   
1    TH002  Miami    S001 2025-08-22 11:28:39.390101            73  68.6   
2    TH003  Miami    S001 2025-08-22 11:28:39.390101            51  69.5   
3    TH004  Miami    S001 2025-08-22 11:28:39.390101            72  70.8   
4    TH005  Miami    S001 2025-08-22 11:28:39.390101            57  70.8   

  Temp_UoM CombinedID  
0        F     S001-1  
1        F     S001-1  
2        F     S001-1  
3        F     S001-1  
4        F     S001-1  


In [ ]:


# # Create Pandas DataFrame
# df = pd.DataFrame(records)

# # Convert to Spark DataFrame
# spark = SparkSession.builder.getOrCreate()


# schema = StructType([
#     StructField("DeviceId", StringType(), True),
#     StructField("City", StringType(), True),
#     StructField("StoreID", StringType(), True),
#     StructField("EnqueuedTimeUTC", TimestampType(), True),
#     StructField("BatteryLevel", IntegerType(), True),
#     StructField("Temp", FloatType(), True),
#     StructField("Temp_UoM", StringType(), True),
#     #StructField("ShiftNo", IntegerType(), True),      # ✅ include this
#     StructField("CombinedID", StringType(), True)     # ✅ include this
# ])
# spark_df = spark.createDataFrame(df, schema=schema)

# Write to Lakehouse table
#spark_df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable("ThermostatRealtime")
#spark_df.write.mode("append").saveAsTable("Thermostat")


StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 38, Finished, Available, Finished)

In [38]:
!pip install azure-eventhub

StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 39, Finished, Available, Finished)

In [ ]:
import json
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime

# Custom JSON encoder to handle datetime
class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()  # Convert datetime to string
        return super().default(obj)

# Event Hub details

CONNECTION_STR = "#Connection string-primary key#"
EVENT_HUB_NAME = "#Event hub name#"

# Initialize Event Hub producer
producer = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR, eventhub_name=EVENT_HUB_NAME)

event_data_batch = producer.create_batch()
for row in spark_df.collect():
    record = row.asDict()
    json_payload = json.dumps(record, cls=DateTimeEncoder)  # ✅ Handles datetime
    event_data_batch.add(EventData(json_payload))

# Send the batch
producer.send_batch(event_data_batch)
producer.close()

print("✅ Data sent to Eventstream endpoint.")

StatementMeta(, 7f608929-833d-4dc8-bbee-6f9efc3918b9, 40, Finished, Available, Finished)

✅ Data sent to Eventstream endpoint.
